# Wildfire Risk and Expected Loss to Housing: Sonoma County
Adapted from the San Francisco template. Same data and method, applied to Sonoma County (county FIPS `097`).

**Why Sonoma is a useful contrast.** San Francisco's tract risks were near zero, so the median was zero and the totals were tiny. Sonoma has large wildland-urban-interface areas (the 2017 Tubbs Fire reached into Santa Rosa neighborhoods), so expect a skewed, much higher-risk distribution where the choice of weights and statistics matters.

Inputs (same as the SF notebook, in `inputdata/`):
1. `stats.csv` from CarbonPlan Open Climate Risk (tract level, 7 metadata rows to skip)
2. `cb_2025_06_tract_500k` California tract shapefile
3. ACS 5-year median home value and owner-occupied homes via `census_helper` (needs `census_api_key.py`)

Reference: O. Chegwidden et al. (2026) "Making climate risk data open." CarbonPlan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import geopandas as gpd
import census_helper

cwd = pathlib.Path().cwd()
inputdir = cwd / 'inputdata'

# Constants for this notebook
OCR_SKIP_ROWS = 7
CA_FIPS = '06'
COUNTY_FIPS = '097'      # Sonoma County.
COUNTY_NAME = 'Sonoma'
MORTGAGE_YEARS = 30
ACS_MISSING_FLAG = -666666666

## Step 1: Load, clean, subset to California

In [ ]:
data = pd.read_csv(inputdata := inputdir / 'stats.csv', skiprows=OCR_SKIP_ROWS)
data['GEOID'] = data['GEOID'].astype(str).str.zfill(11)
data[['rps_2011_mean', 'rps_2047_mean']] /= 100   # percent -> fraction per year

ca_risk = (data[data['GEOID'].str.startswith(CA_FIPS)]
           .set_index('GEOID')[['rps_2011_mean', 'rps_2047_mean', 'building_count']])
print(len(ca_risk), 'CA tracts;', round(ca_risk['building_count'].sum() / 1e6, 2), 'M buildings')

## Step 2: Merge with tract geometry and subset to Sonoma

In [ ]:
gdf = gpd.read_file(inputdir / 'cb_2025_06_tract_500k' / 'cb_2025_06_tract_500k.shp')
county_gdf = gdf[gdf['COUNTYFP'] == COUNTY_FIPS]
print(len(county_gdf), 'census tracts in', COUNTY_NAME)

county = (county_gdf.merge(ca_risk, left_on='GEOID', right_index=True, how='inner')
          .set_index('GEOID')[['rps_2011_mean', 'rps_2047_mean', 'building_count', 'ALAND', 'geometry']])
print(len(county), 'tracts with risk data;', int(county['building_count'].sum()), 'modeled buildings')
county.describe()

In SF the median tract risk was zero. Check whether that is true here.

In [ ]:
county[['rps_2011_mean', 'rps_2047_mean']].describe(percentiles=[.25, .5, .75, .9])

In [ ]:
county.explore(column='rps_2047_mean', tiles='CartoDB positron', cmap='Oranges')

## Step 3: Merge ACS owner-occupied homes and median home value
Negative values (`-666666666`) are the ACS missing-data flag; the median is also top-coded near \$2M, which matters much less in Sonoma than in SF but still understates the most expensive tracts.

In [ ]:
variables = {'B25077_001E': 'Median owner-occupied home value ($)',
             'B25003_002E': 'Owner-occupied homes'}
VAL, HOMES = list(variables.values())

acs = census_helper.retrieve_acs_var(variables, geo='tract')
acs_valid = acs[(acs[VAL] != ACS_MISSING_FLAG) & (acs[VAL] > 0) & (acs[HOMES] >= 1)]

df = county.merge(acs_valid[[VAL, HOMES]], left_index=True, right_index=True, how='inner')

dropped = acs.loc[acs.index.intersection(county.index).difference(df.index)]
print(f'Kept {len(df)} of {len(county)} tracts.')
print(f'Dropped {len(dropped)} tracts with {dropped[HOMES].clip(lower=0).sum():,} owner-occupied homes '
      f'out of {df[HOMES].sum() + dropped[HOMES].clip(lower=0).sum():,}.')

## Step 4: Risk, exposure, and insurance statistics
Because Sonoma's distribution is skewed, use **home-weighted** statistics (each tract counts in proportion to its owner-occupied homes) instead of simple tract means.

In [ ]:
def compound_risk(x, n=MORTGAGE_YEARS):
    # Probability of losing the structure at least once in n years (x is a fraction per year)
    return 1 - (1 - x) ** n

def wavg(values, weights):
    return (values * weights).sum() / weights.sum()

def wquantile(values, weights, q):
    order = np.argsort(values.values)
    v, w = values.values[order], weights.values[order]
    cum = np.cumsum(w) / w.sum()
    return v[np.searchsorted(cum, q)]

for yr in ('2011', '2047'):
    df[f'compound_risk{yr}'] = df[f'rps_{yr}_mean'].map(compound_risk)
    df[f'expected_annual_loss{yr}'] = df[f'rps_{yr}_mean'] * df[VAL] * df[HOMES]   # $/yr, whole tract
    df[f'premium_per_home{yr}'] = df[f'rps_{yr}_mean'] * df[VAL]                   # $/yr, break-even

summary = pd.DataFrame({
    '2011': [wavg(df['rps_2011_mean'], df[HOMES]), wquantile(df['rps_2011_mean'], df[HOMES], .5),
             wquantile(df['rps_2011_mean'], df[HOMES], .9), df['rps_2011_mean'].max()],
    '2047': [wavg(df['rps_2047_mean'], df[HOMES]), wquantile(df['rps_2047_mean'], df[HOMES], .5),
             wquantile(df['rps_2047_mean'], df[HOMES], .9), df['rps_2047_mean'].max()],
}, index=['home-weighted mean', 'home-weighted median', 'home-weighted 90th pct', 'worst tract'])
summary.style.format('{:.2e}')

In [ ]:
print('Annual risk, home-weighted mean 2011 -> 2047: '
      f"{summary.loc['home-weighted mean','2011']:.2e} -> {summary.loc['home-weighted mean','2047']:.2e}")
print(f'{MORTGAGE_YEARS}-yr chance of loss, typical (weighted-mean) home, 2047 climate: '
      f"{compound_risk(summary.loc['home-weighted mean','2047']):.2%}")
print(f'{MORTGAGE_YEARS}-yr chance of loss, worst tract, 2047 climate: '
      f"{compound_risk(summary.loc['worst tract','2047']):.1%}")

### Minimum break-even insurance premium and total exposure

In [ ]:
worst = df['rps_2047_mean'].idxmax()
print('Highest-risk tract:', worst)
print(df.loc[worst, [VAL, HOMES, 'rps_2047_mean', 'compound_risk2047', 'premium_per_home2047']])

# Total expected loss over the mortgage horizon (structure lost at most once)
total_exposure = {yr: (df[f'compound_risk{yr}'] * df[VAL] * df[HOMES]).sum() for yr in ('2011', '2047')}
total_homes_value = (df[VAL] * df[HOMES]).sum()
for yr, v in total_exposure.items():
    print(f'{yr} climate: expected {MORTGAGE_YEARS}-yr loss ${v/1e9:.2f}B '
          f'({v/total_homes_value:.2%} of total owner-occupied value; ~${v/MORTGAGE_YEARS/df[HOMES].sum():,.0f}/home/yr)')

### Where is the exposure concentrated?

In [ ]:
df['exposure2047'] = df['compound_risk2047'] * df[VAL] * df[HOMES]
top = df.sort_values('exposure2047', ascending=False).head(10)
top[[HOMES, VAL, 'rps_2047_mean', 'compound_risk2047', 'premium_per_home2047', 'exposure2047']]

In [ ]:
share_top10 = top['exposure2047'].sum() / df['exposure2047'].sum()
print(f'Top 10 tracts ({len(top)/len(df):.0%} of tracts) hold {share_top10:.0%} of total 2047 exposure')

fig, ax = plt.subplots(figsize=(7, 4))
top['exposure2047'].div(1e6).iloc[::-1].plot.barh(ax=ax, color='tab:orange')
ax.set_xlabel(f'Expected {MORTGAGE_YEARS}-yr loss, 2047 climate ($M)')
ax.set_ylabel('Census tract GEOID')
plt.tight_layout()

In [ ]:
df.explore(column='exposure2047', tiles='CartoDB positron', cmap='Reds')

### Does risk rise with climate change, and who bears it?
Compare 2011 vs 2047 and group tracts into risk quartiles (home-weighted), the Sonoma analogue of the SF "expensive vs not" pivot, since here risk varies enough to be the natural grouping variable.

In [ ]:
df['change_2011_2047'] = df['rps_2047_mean'] - df['rps_2011_mean']
share_worse = df.loc[df['change_2011_2047'] > 0, HOMES].sum() / df[HOMES].sum()
print(f'{share_worse:.0%} of owner-occupied homes are in tracts where risk rises by 2047')

df['risk_quartile'] = pd.qcut(df['rps_2047_mean'], 4, labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'], duplicates='drop')
by_q = df.groupby('risk_quartile', observed=True).apply(lambda g: pd.Series({
    'tracts': len(g),
    'owner homes': g[HOMES].sum(),
    'median home value ($, home-weighted)': wquantile(g[VAL], g[HOMES], .5),
    'mean annual risk (home-weighted)': wavg(g['rps_2047_mean'], g[HOMES]),
    'break-even premium ($/home/yr)': wavg(g['premium_per_home2047'], g[HOMES]),
}))
by_q

## Caveats (carry over from the SF template, with Sonoma-specific notes)
- The model does not explicitly capture urban conflagrations, so Santa Rosa-type neighborhoods are likely **under**estimated.
- Tract medians stand in for parcel-level values; in large rural tracts risk and home value can be correlated within the tract, biasing totals.
- Home value includes land, overstating structure replacement cost; the ACS median is also top-coded.
- Risks are for an unmitigated generic structure; defensible space, retrofits and building age would lower them.
- 2047 uses an RCP8.5 scenario, now considered a high-end case.